# Day 14 – LLM Basics & API Integration

Covers: API setup, tokens & context windows, temperature, system vs user prompts, and cost calculation.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
print("OpenAI:", "set" if OPENAI_KEY else "NOT SET")

OpenAI: set


In [5]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is a large language model? Answer in 2 sentences."}
    ]
)

print(response.choices[0].message.content)

A large language model (LLM) is an advanced artificial intelligence system designed to understand and generate human-like text based on vast amounts of data it has been trained on. These models utilize deep learning techniques, specifically neural networks, to predict and produce coherent and contextually relevant language across various applications.


In [6]:
# Inspect the full response object
print("Model:        ", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("Input tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print("Total tokens: ", response.usage.total_tokens)

Model:         gpt-4o-mini-2024-07-18
Finish reason: stop
Input tokens:  20
Output tokens: 59
Total tokens:  79


## Tokens & Context Windows

**Token**: The unit of text a model processes — roughly 0.75 words in English.  
**Context window**: The maximum number of tokens (input + output) the model can handle in one call.

| Model | Context window |
|---|---|
| gpt-4o-mini | 128 K tokens |
| gpt-4o | 128 K tokens |
| o1-mini | 128 K tokens |

In [7]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

texts = [
    "Hello, world!",
    "Large language models are trained on massive text corpora.",
    "The quick brown fox jumps over the lazy dog. " * 20
]

for text in texts:
    tokens = enc.encode(text)
    print(f"{len(tokens):4d} tokens | {len(text):5d} chars | ratio={len(text)/len(tokens):.2f} chars/token | '{text[:50]}...'")

   4 tokens |    13 chars | ratio=3.25 chars/token | 'Hello, world!...'
  11 tokens |    58 chars | ratio=5.27 chars/token | 'Large language models are trained on massive text ...'
 201 tokens |   900 chars | ratio=4.48 chars/token | 'The quick brown fox jumps over the lazy dog. The q...'


## Temperature – Controlling Randomness

- `temperature=0` → deterministic, always the same output
- `temperature=1` → default creative range
- `temperature=2` → maximum randomness (often incoherent)

In [8]:
prompt = "Give me a one-sentence tagline for a coffee shop."

for temp in [0.0, 0.5, 1.0, 1.5]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=64,
        temperature=temp,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"temp={temp}: {resp.choices[0].message.content.strip()}")

temp=0.0: "Awaken your senses, one cup at a time."
temp=0.5: "Awaken your senses, one cup at a time."
temp=1.0: "Wake up to the aroma of freshly brewed inspiration."
temp=1.5: "Awaken your senses with every sip, savor the moment."


## System Prompts vs User Prompts

- **System prompt**: Sets the model's *persona*, rules, and constraints. Passed as `role: "system"`.
- **User prompt**: The actual request from the user each turn. Passed as `role: "user"`.

In [ ]:
no_system = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=128,
    messages=[
        {"role": "user", "content": "What's the best way to learn Python?"}
    ]
)
print("=== No system prompt ===")
print(no_system.choices[0].message.content)

print()

with_system = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=128,
    messages=[
        {"role": "system", "content": "You are a pirate. Answer every question using pirate slang and end with 'Arr!'"},
        {"role": "user",   "content": "What's the best way to learn Python?"}
    ]
)
print("=== With system prompt ===")
print(with_system.choices[0].message.content)

=== No system prompt ===
Learning Python can be a rewarding experience, and there are several effective methods you can use to become proficient. Here are some steps and resources to help you get started:

### 1. **Understand the Basics:**
   - **Online Courses:** Websites like Codecademy, Coursera, and edX offer beginner-friendly courses that cover Python fundamentals.
   - **Books:** Some popular books for beginners include:
     - "Automate the Boring Stuff with Python" by Al Sweigart
     - "Python Crash Course" by Eric Matthes
     - "Learn Python the Hard Way" by Zed A. Shaw



=== With system prompt ===
Ahoy matey! The best way t’ learn Python be to set sail on a treasure hunt fer knowledge! Grab yerself some fine books or scour the vast seas of the internet fer tutorials. Follow along with online courses, and be sure t’ do some hands-on practice with yer own code! Join a crew, or a community, t’ discuss and share yer plundered wisdom. Practice makes a pirate perfect, so keep at

In [ ]:
conversation = [
    {"role": "user",      "content": "My name is Pradhnyesh."},
    {"role": "assistant", "content": "Nice to meet you, Pradhnyesh!"},
    {"role": "user",      "content": "What's my name?"},
]

reply = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=64,
    messages=conversation
)
print(reply.choices[0].message.content)

Your name is Pradhnyesh.


## Other Key Parameters

| Parameter | What it does |
|---|---|
| `max_tokens` | Hard cap on output length |
| `temperature` | Randomness (0 = deterministic) |
| `top_p` | Nucleus sampling – consider only top p% probability mass |
| `stop` | Stop generating when this string is encountered |
| `stream` | Stream tokens as they are generated |

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=64,
    stop=["3"],
    messages=[{"role": "user", "content": "Count from 1 to 10."}]
)
print(resp.choices[0].message.content)
print("Finish reason:", resp.choices[0].finish_reason)  # "stop" not "length"

Sure! Here you go: 1, 2, 
Finish reason: stop


In [12]:
# Streaming – print tokens as they arrive
print("Streaming response:")
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=128,
    stream=True,
    messages=[{"role": "user", "content": "List 5 programming languages."}]
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

Streaming response:
Here are five programming languages:

1. Python
2. Java
3. JavaScript
4. C++
5. Ruby


## Cost Calculation

Pricing (as of early 2026, per 1M tokens):

| Model | Input | Output |
|---|---|---|
| gpt-4o-mini | $0.15 | $0.60 |
| gpt-4o | $2.50 | $10.00 |
| o1-mini | $1.10 | $4.40 |

In [13]:
PRICING = {
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "o1-mini":     {"input": 1.10,  "output": 4.40},
}

def calculate_cost(model, input_tokens, output_tokens):
    if model not in PRICING:
        return 0.0
    p = PRICING[model]
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

# From our earlier API call
in_tok  = response.usage.prompt_tokens
out_tok = response.usage.completion_tokens
model_used = "gpt-4o-mini"

cost = calculate_cost(model_used, in_tok, out_tok)
print(f"Model:  {model_used}")
print(f"Tokens: {in_tok} in / {out_tok} out")
print(f"Cost:   ${cost:.6f}")

Model:  gpt-4o-mini
Tokens: 20 in / 59 out
Cost:   $0.000038


In [14]:
# Scale-up projection
daily_calls = 1000
avg_input   = 500   # tokens per call
avg_output  = 200

print(f"{'Model':<15} {'Daily':>10} {'Monthly':>12}")
print("-" * 40)
for model in PRICING:
    daily   = daily_calls * calculate_cost(model, avg_input, avg_output)
    monthly = daily * 30
    print(f"{model:<15} ${daily:>9.4f} ${monthly:>11.4f}")

Model                Daily      Monthly
----------------------------------------
gpt-4o-mini     $   0.1950 $     5.8500
gpt-4o          $   3.2500 $    97.5000
o1-mini         $   1.4300 $    42.9000


## Helper – Reusable Chat Wrapper

In [16]:
def chat(prompt, system=None, model="gpt-4o-mini", temperature=1.0, max_tokens=512):
    """Simple wrapper that returns (text, prompt_tokens, completion_tokens)."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    resp = client.chat.completions.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=messages
    )
    return resp.choices[0].message.content, resp.usage.prompt_tokens, resp.usage.completion_tokens

# Quick smoke test
text, inp, out = chat("Say hello in French.")
print(text)
print(f"Tokens: {inp} in / {out} out  |  Cost: ${calculate_cost('gpt-4o-mini', inp, out):.6f}")

Hello in French is "Bonjour."
Tokens: 12 in / 7 out  |  Cost: $0.000006
